## Spark session configuration

In [ ]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
        .appName('data processing')
        # 1. Compatibility & Performance
        .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
        .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
        .config('spark.scheduler.mode', 'FAIR')
        
        # 2. Adaptive Query Execution (Fixes Tiny Files)
        .config('spark.sql.adaptive.enabled', 'true')
        .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
        .config('spark.sql.adaptive.coalescePartitions.minPartitionNum', '1')
        
        # 3. Delta Optimization
        .config("spark.microsoft.delta.vorder.enabled", "true")
        .config('spark.microsoft.delta.optimizeWrite.enabled', 'true')
        .config('spark.microsoft.delta.autoCompact.enabled', 'true')
        .config('spark.microsoft.delta.optimizeWrite.binSize', '134217728') # 128MB Target

        # 4. Delta Versioning & Deletion Vectors
        .config('spark.databricks.delta.minReaderVersion', '3')          
        .config('spark.databricks.delta.minWriterVersion', '7')          
        .config("spark.databricks.delta.properties.defaults.enableDeletionVectors", "true")
        # 5. Limit Control
        .config('spark.sql.files.maxRecordsPerFile', '0')

        .config('spark.databricks.delta.optimizeMetadataQuery.enabled', 'true')

        .getOrCreate()
)

## Environment configuration

In [ ]:
from delta.tables import DeltaTable
import hashlib
from datetime import datetime
from typing import List, Optional, Literal
import re

In [ ]:
force_rebuild = False # If True - force MLV deletion and creation from scratch
debug_mode = True

source_ws = 'dev_demo_core'
target_ws = 'dev_demo_core'

print(f"Run settings: 'force_rebuild' - {force_rebuild} | 'debug_mode' = {debug_mode}")
print(f"Workspace references: 'source_ws' - {source_ws} | 'target_ws' = {target_ws}")

In [ ]:
def build_onelake_path(workspace: str, lakehouse: str, section: Literal["Tables", "Files"] = "Tables",
    schema: str | None = None, table: str | None = None, folder_path: str | None = None, ) -> str:
    """Builds a OneLake ABFSS path for either the "Tables" or "Files" section.

    Args:
        workspace: The name of the target Fabric workspace.
        lakehouse: The name of the target Lakehouse.
        section: The storage section, strictly "Tables" or "Files". Defaults to "Tables".
        schema: Optional schema name for namespaced tables.
        table: Optional target table name.
        folder_path: Optional subfolder path when accessing the "Files" section.

    Returns:
        A fully constructed ABFSS path string for OneLake access.
        
    Raises:
        ValueError: If a schema is provided but no target table is specified.
    """
    # Define the foundational OneLake URI using workspace and lakehouse identifiers
    base_path = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{lakehouse}.Lakehouse/{section}"

    if section == "Tables":
        if schema and not table:
            raise ValueError("A table must be specified if a schema is provided.")
        
        if schema and table:
            return f"{base_path}/{schema}/{table}"
        if table:
            return f"{base_path}/{table}"

    elif section == "Files" and folder_path:
        # Strip leading slashes to prevent double-slash malformed URIs
        clean_folder = folder_path.lstrip("/")
        return f"{base_path}/{clean_folder}"

    # Return the base path if no specific granular item or folder was requested
    return base_path


def text_to_hash(text: str | None) -> str:
    """
    Normalizes and hashes a given text string using MD5.
    
    The normalization process strips leading/trailing whitespace, condenses 
    multiple sequential whitespaces into a single space, and converts the 
    string to lowercase.
    
    Args:
        text: The input string to be hashed. Can be None or an empty string.
        
    Returns:
        The MD5 hex digest of the normalized string, or the literal string 
        "None" if the input is falsy (explicitly required for Delta table 
        comment parameters).
    """
    if not text: 
        return "None"
    # Convert all whitespace (newlines, tabs, multiple spaces) into a single space, and lowercase
    normalized_text = re.sub(r'\s+', ' ', text.strip())
    return hashlib.md5(normalized_text.encode('utf-8')).hexdigest()


def deploy_mlv(spark: SparkSession, table_name: str, schema_name: str, lakehouse_name: str, workspace_name: str, select_query: str, 
    constraints: Optional[str] = None, partition_by: Optional[str] = None, cluster_by: Optional[str] = None, force_rebuild: bool = False, 
    comment: Optional[str] = None, description: Optional[str] = None, debug: bool = False ) -> None:
    """
        Standardized Fabric MLV Deployment Wrapper.
        
        Manages the lifecycle of Materialized Lake Views natively using only 
        CREATE OR REPLACE or REFRESH operations to comply with Fabric.
        Enforces Medallion Gold standards (Deletion Vectors, CDF) and handles 
        smart state-checking to avoid unnecessary rebuilds.

        :param spark: Active PySpark SparkSession object.
        :param table_name: Target name of the Materialized Lake View (e.g., 'dim_date').
        :param schema_name: Target schema name (e.g., 'dbo').
        :param lakehouse_name: Target lakehouse artifact name.
        :param workspace_name: Target workspace name. Enclosed in backticks safely during execution.
        :param select_query: The core SQL SELECT statement defining the view's logic.
        :param constraints: Optional Delta constraints (e.g., 'date IS NOT NULL'). 
                            Changes to this trigger a CREATE OR REPLACE.
        :param partition_by: Optional comma-separated string of columns for Hive-style partitioning.
                            Mutually exclusive with cluster_by.
        :param cluster_by: Optional comma-separated string of columns for Liquid Clustering (V-Order).
                        Mutually exclusive with partition_by.
        :param force_rebuild: If True, bypasses state checks and forces a CREATE OR REPLACE.
                            Useful for manual pipeline resets.
        :param comment: High-level table comment. Maps to Delta's native 'COMMENT' TBLPROPERTY.
        :param description: Detailed table description. Maps to Fabric's specific 'COMMENTS' / 'comment' 
                            TBLPROPERTIES to ensure visibility in the Fabric UI/Catalog.
        :param debug: If True, prints verbose trace logs showing hash comparisons and state logic.
        """
    start_time = datetime.now()

    # Clean optional string parameters for safe evaluation
    constraints = constraints.strip() if constraints else None
    partition_by = partition_by.strip() if partition_by else None
    cluster_by = cluster_by.strip() if cluster_by else None
    
    # Define exact DESIRED state (treating None and "" as identical empty states)
    desired_comment = comment.strip() if comment else ""
    desired_desc = description.strip() if description else ""
    desired_query_hash = text_to_hash(select_query)
    desired_constraints_hash = text_to_hash(constraints) if constraints else text_to_hash('None')

    # Track storage strategies as exact strings for property comparison
    desired_partition_state = partition_by if partition_by else 'None'
    desired_cluster_state = cluster_by if cluster_by else 'None'

    # 1. Formatting for SQL
    schema_full_name = f"`{workspace_name}`.{lakehouse_name}.{schema_name}"
    mlv_full_name = f"{schema_full_name}.{table_name}"
    mlv_path = build_onelake_path(workspace=workspace_name, lakehouse=lakehouse_name, schema=schema_name, table=table_name)
    
    # 2. Storage Strategy logic
    storage_clause = ""
    if cluster_by:
        storage_clause = f"CLUSTER BY ({cluster_by})"
    elif partition_by:
        storage_clause = f"PARTITIONED BY ({partition_by})"

    # 3. TBLPROPERTIES Generation (Fabric-specific mapping)
    tbl_props = []
    if desired_comment:
        tbl_props.append(f"'COMMENT' = '{desired_comment}'")
        
    if desired_desc:
        tbl_props.append(f"'comment' = '{desired_desc}'")
        tbl_props.append(f"'COMMENTS' = '{desired_desc}'")

    # Modern Delta Lake features for Fabric
    tbl_props.append("'delta.enableDeletionVectors' = 'true'")
    tbl_props.append("'delta.enableChangeDataFeed' = 'true'")
    tbl_props.append("'delta.autoOptimize.autoCompact' = 'true'")
    tbl_props.append("'delta.parquet.vorder.enabled' = 'true'")
    # Hashes
    tbl_props.append(f"'mlv.query_hash' = '{desired_query_hash}'")
    tbl_props.append(f"'mlv.constraints_hash' = '{desired_constraints_hash}'")
    
    # Append storage states to properties for future state-checking
    tbl_props.append(f"'mlv.partition_by' = '{desired_partition_state}'")
    tbl_props.append(f"'mlv.cluster_by' = '{desired_cluster_state}'")

    # 4. Create Statement Generation
    constraints_clause = f"({constraints})" if constraints else ""
    create_sql = f"""
        CREATE OR REPLACE MATERIALIZED LAKE VIEW {mlv_full_name}
        {constraints_clause}
        {storage_clause}
        TBLPROPERTIES ({', '.join(tbl_props)})
        AS {select_query}
    """
    
    try:
        # 5. Ensure Schema Exists
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_full_name}")

        exists = notebookutils.fs.exists(mlv_path)
        requires_replace = False
        
        # 6. Advanced Existence & State Check
        if exists:
            props_df = spark.sql(f"SHOW TBLPROPERTIES {mlv_full_name}").collect()
            
            # CRITICAL: We cannot use .lower() here because Fabric relies on 'COMMENT' vs 'comment'
            current_props = {row['key']: row['value'] for row in props_df}
            
            # Extract CURRENT state
            current_query_hash = current_props.get('mlv.query_hash', "")
            current_constraints_hash = current_props.get('mlv.constraints_hash')
            current_partition_state = current_props.get('mlv.partition_by', 'None')
            current_cluster_state = current_props.get('mlv.cluster_by', 'None')
            
            # Match Fabric's exact casing
            current_comment = current_props.get('COMMENT', "").strip()
            # If description maps to both, checking one is sufficient for the state validation
            current_desc = current_props.get('COMMENTS', "").strip() 
            
            if debug:
                print(f"🔍 [DEBUG] Hash Check: New [{desired_query_hash}] vs Old [{current_query_hash}]")
                print(f"🔍 [DEBUG] Constraint Check: New [{desired_constraints_hash}] vs Old [{current_constraints_hash}]")
                print(f"🔍 [DEBUG] Partition Check: New [{desired_partition_state}] vs Old [{current_partition_state}]")
                print(f"🔍 [DEBUG] Cluster Check: New [{desired_cluster_state}] vs Old [{current_cluster_state}]")
                print(f"🔍 [DEBUG] Comment Check: New [{desired_comment}] vs Old [{current_comment}]")
                print(f"🔍 [DEBUG] Desc Check: New [{desired_desc}] vs Old [{current_desc}]")

            # Compare desired vs current state
            if desired_query_hash != current_query_hash:
                print("⚠️ Query logic mismatch detected. Triggering CREATE OR REPLACE.")
                requires_replace = True
            if desired_constraints_hash != current_constraints_hash:
                print("⚠️ Constraints mismatch detected. Triggering CREATE OR REPLACE.")
                requires_replace = True
            elif desired_partition_state != current_partition_state:
                print(f"⚠️ Partitioning mismatch detected. Triggering CREATE OR REPLACE. ('{current_partition_state}' -> '{desired_partition_state}')")
                requires_replace = True
            elif desired_cluster_state != current_cluster_state:
                print(f"⚠️ Clustering mismatch detected. Triggering CREATE OR REPLACE. ('{current_cluster_state}' -> '{desired_cluster_state}')")
                requires_replace = True
            elif desired_comment != current_comment:
                print(f"⚠️ Comment mismatch. Triggering CREATE OR REPLACE. ('{current_comment}' -> '{desired_comment}')")
                requires_replace = True
            elif desired_desc != current_desc:
                print(f"⚠️ Description mismatch. Triggering CREATE OR REPLACE. ('{current_desc}' -> '{desired_desc}')")
                requires_replace = True

        # 7. Execution
        if force_rebuild or requires_replace:
            if debug: print(f"🧨 [DEBUG] Triggering DROP/CREATE for {mlv_full_name}")
            spark.sql(f"DROP MATERIALIZED LAKE VIEW IF EXISTS {mlv_full_name}")
            result_df = spark.sql(create_sql)
            status = "Rebuilt"
        elif exists:
            if debug: print(f"🔄 [DEBUG] Triggering REFRESH for {mlv_full_name}")
            result_df = spark.sql(f"REFRESH MATERIALIZED LAKE VIEW {mlv_full_name}")
            status = "Refreshed"
        else:
            if debug: print(f"🚀 [DEBUG] Triggering CREATE for {mlv_full_name}")
            result_df = spark.sql(create_sql)
            status = "Deployed"

        duration = (datetime.now() - start_time).total_seconds()
        
        # 8. Metrics Extraction
        metrics: Dict[str, Any] = {}
        if result_df:
            summary = result_df.collect()
            metrics = {row['metric_name']: row['metric_value'] for row in summary if 'metric_name' in row}
        
        refresh_policy = metrics.get('RefreshPolicy', 'Unknown')
        rows_processed = metrics.get('TotalRowsProcessed', '0')
        msg = metrics.get('Message', 'No message provided.')
        
        if status == "Refreshed" and refresh_policy == "NoRefresh":
            status += " (Skipped - No Source Changes)"

        # Clean Output
        print(f"✅ SUCCESS: {mlv_full_name} | {status} | {duration:.2f}s")
        print(f"   ├─ Action: {refresh_policy} | Rows Processed: {rows_processed} | Message: {msg}")
        
        if debug:
            print(f"   └─ Raw Metrics: {metrics}")
            
    except Exception as e:
        print(f"🔥 Error during MLV deployment for {mlv_full_name}: {str(e)}")
        raise

In [ ]:
%%sql
SELECT 
    Namespace,
    MLVName,
    RefreshPolicy,   -- Identifies if Fabric ran a FullRefresh, Incremental, or NoRefresh
    RefreshDate,
    TotalRowsProcessed,
    TotalRowsDropped, -- Rows eliminated by ON MISMATCH DROP constraints
    ViolationsPerConstraint
FROM `dev_demo_core`.lh_gold.dbo.sys_dq_metrics
WHERE RefreshDate = CURRENT_DATE()
ORDER BY RefreshTimestamp DESC

## Dimensions

In [ ]:
target_table  = "dim_date"
target_schema = "dev"
target_lh     = "lh_gold"

description = "Description - Dates dimension"
# Generic table comment
comment = "Dates dimension"

constraints = """ """
partition_by = " "

select_query = f"""
    WITH bounds AS (
        SELECT
            DATE_TRUNC('month', MIN(dtg_dt)) AS min_dt
            , LAST_DAY(MAX(dtg_dt)) AS max_dt
        FROM {source_ws}.lh_silver.demo.weather_observation
    )
    , calendar AS (
        -- Use the values from the tiny 'bounds' table
        SELECT SEQUENCE(min_dt, max_dt, INTERVAL 1 DAY) AS dates
        FROM bounds
    )
    , dates_flat AS (
        SELECT EXPLODE(dates) AS date
        FROM calendar
    )
    , date_dim AS (
        SELECT
            -- Date ID: Format YYYYMMDD as an Integer
            CAST(DATE_FORMAT(date, 'yyyyMMdd') AS INTEGER) AS date_id
            , date AS date
            , YEAR(date) AS year
            , CONCAT('Q', QUARTER(date)) AS quarter
            , MONTH(date) AS month_number
            , DATE_FORMAT(date, 'MMMM') AS Month
            , WEEKOFYEAR(date) AS week
            , DAY(date) AS day
            , WEEKOFYEAR(date) AS iso_week
            -- Week Start (Monday) and End (Sunday)
            , DATE_SUB(NEXT_DAY(date, 'Monday'), 7) AS week_start_day
            , DATE_ADD(DATE_SUB(NEXT_DAY(date, 'Monday'), 7), 6) AS week_end_day
            , CAST(DATE_FORMAT(date, 'yyyyMM') AS INTEGER) AS yyyymm
        FROM dates_flat
    )

    SELECT *
    FROM date_dim d

    """

deploy_mlv(spark, workspace_name = target_ws, table_name = target_table, schema_name = target_schema, lakehouse_name = target_lh, select_query = select_query,
            constraints = constraints, partition_by=partition_by, force_rebuild=force_rebuild, comment=comment, description=description, debug=debug_mode)

In [ ]:
target_table  = "dim_weather_station"
target_schema = "dim"
target_lh     = "lh_gold"

description = "Weather Station Dimension: Generate unique station identifiers from weather observation locations"
comment = "Weather Station Dimension"

constraints = None

select_query = f"""
    SELECT DISTINCT
        hash(wo.location) AS station_id      -- Hash-based unique identifier
        , wo.location AS station
        , wo.location_name AS station_name
        , wo.latitude
        , wo.longitude
        , wo.altitude
    FROM {source_ws}.lh_silver.demo.weather_observation AS wo
    """

deploy_mlv(spark, workspace_name = target_ws, table_name = target_table, schema_name = target_schema, lakehouse_name = target_lh, select_query = select_query,
            constraints = constraints, partition_by=partition_by, force_rebuild=force_rebuild, comment=comment, description=description, debug=debug_mode)

In [ ]:
%%sql
SHOW TBLPROPERTIES lh_gold.dim.dim_weather_station

## Facts

In [ ]:
target_table  = "fact_weather_measurements"
target_schema = "fact"
target_lh     = "lh_gold"

description = "Weather Measurements Fact: Extract weather measurements with surrogate keys for dimensional modeling"
comment = "Fact - Weather Measurements"

constraints = """
    CONSTRAINT temperature_not_null CHECK (temperature is not null) ON MISMATCH DROP
    """
partition_by = "station_key"

select_query = f"""
    SELECT
        dtg AS measurement_ts
        , hash(wo.location) AS station_key
        , CAST(DATE_FORMAT(wo.dtg_dt, 'yyyyMMdd') AS INTEGER) AS date_key     -- Date surrogate key (YYYYMMDD format)
        , CAST(DATE_FORMAT(wo.dtg, 'HHmm') AS INTEGER) AS time_key            -- Time surrogate key (HHMM format)
        , t AS temperature                                             -- Dry temperature (unit: likely °C)
        , t_d AS temperature_dewpoint                                    -- Dew point temperature
        , h AS humidity                                                     -- Relative humidity (unit: likely %).
    FROM {source_ws}.lh_silver.demo.weather_observation AS wo
    """

deploy_mlv(spark, workspace_name = target_ws, table_name = target_table, schema_name = target_schema, lakehouse_name = target_lh, select_query = select_query,
            constraints = constraints, partition_by=partition_by, force_rebuild=force_rebuild, comment=comment, description=description, debug=debug_mode)

In [ ]:
%%sql
describe detail lh_gold.fact.fact_weather_measurements

In [ ]:
%%sql
SHOW TBLPROPERTIES lh_gold.fact.fact_weather_measurements

In [ ]:
%%sql
DESCRIBE EXTENDED lh_gold.fact.fact_weather_measurements